# C11-neural-training — Losses, Gradients, and Training (Unit Overview)

This is the index notebook for the five 90-minute teaching sessions in
`lessons/`. C5 supplied network architecture and forward passes; C6 supplied
PyTorch tensors, `nn.Module`, parameters, and inspection. C11 now joins those
ideas into a complete, auditable training system.

## Unit goals

By the end of the unit, you can:

- turn a batch of logits of shape `(N, C)` into stable softmax probabilities,
  evaluate categorical cross-entropy, derive the softmax Jacobian, and derive
  the fused logit gradient `(p - y) / N`;
- backpropagate through a two-layer MLP by hand, with every intermediate shape
  and every accumulated gradient stated;
- implement and certify a seeded NumPy MLP training loop from forward cache
  through backward pass and parameter update;
- train the same model with PyTorch autograd and an optimizer, using the exact
  lifecycle `zero_grad → forward → loss → backward → step`;
- distinguish parameters from buffers and training behavior from deterministic
  evaluation behavior for BatchNorm and dropout.

## Prerequisites recap

The declared prerequisites are **F4-multivar-calculus**,
**C3-gradient-descent**, **C5-neural-networks**, and **C6-pytorch**.

From F4, recall partial derivatives, gradients, and the multivariable chain
rule. From C3, recall learning rate, stochastic gradient descent, loss
surfaces, and the update `parameter -= learning_rate * gradient`. From C5,
recall ReLU, MLP architecture, matrix shapes, initialization variance,
overfitting, and L2 regularization. From C6, recall torch tensors,
`nn.Module`, `nn.Parameter`, `requires_grad`, parameter counting, and
Python inheritance. F1/F5 vocabulary used below includes seeded NumPy arrays,
broadcasting, aggregation axes, expectation, and variance.

## Five-session map

| Session | Notebook | Main contract |
|---|---|---|
| 1 | `01-softmax-and-cross-entropy.ipynb` | stable logits → probabilities → loss, Jacobian, fused gradient |
| 2 | `02-manual-backpropagation.ipynb` | dependency order, shape ledger, accumulation, complete two-layer backward pass |
| 3 | `03-numpy-mlp-training.ipynb` | cache → backward → update, gradient check, trained-vs-untrained behavior |
| 4 | `04-pytorch-autograd-and-optimizers.ipynb` | autograd lifecycle, accumulation, optimizer state, deterministic evaluation |
| 5 | `05-batchnorm-and-dropout.ipynb` | batch/running statistics, affine state, inverted dropout, train/eval audit |

Every assertion that compares floating-point values names `atol` and `rtol`.
Every random example uses seed **20260804**. All code is CPU-small and uses no
download, pretrained model, external API, or hidden data.

## How to study

1. Before running a code cell, predict its shapes and at least one numerical
   invariant.
2. Answer each checkpoint on paper. Answers are collected only at the end of
   that session.
3. Keep a four-column audit beside every training loop: **mode**, **gradient
   state**, **optimizer action**, **evaluation context**.
4. After Session 5, use `review.ipynb` without reopening the lessons; then
   return to the lesson named in the weak-spot table.

## Forward link

C7 reuses this complete training lifecycle for convolutional networks.
Convolution changes the feature extractor and its shape arithmetic; it does
not change the loss, backpropagation logic, optimizer ordering, BatchNorm/
dropout mode contract, or evaluation discipline taught here.


In [ ]:
import numpy as np
import torch

SEED = 20260804
np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("seed:", SEED, "| default torch dtype:", torch.get_default_dtype())